# 01. Full guide (CPU slice)

The four public sections, the lifecycle, exact/QAOA/RQAOA, generators/plans, save/checkpoint. GPU and the still-open Laboratory controllers are marked separately.

In [1]:
import json, tempfile
from pathlib import Path
from itertools import combinations
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from QA_max_cut_lib import WeightedGraph, ExactCPU, QAOACPU, RQAOACPU, RunControl, restore
from QA_max_cut_lib.generators import GraphFactory
from QA_max_cut_lib.generators.plans import build_overlap_edge_partition, build_connected_bfs_partition
from QA_max_cut_lib.tools.partition_analysis import analyze_plan
ROOT=Path.cwd()
if not (ROOT/'configs').exists(): ROOT=ROOT.parent

## Exact reference

In [2]:
graph=WeightedGraph([(10,30,2),(30,80,-1),(10,80,3)],nodes=[10,30,80,101])
exact=ExactCPU(); sol=exact.solve(graph,report=0); print(sol, exact.cut_value)

{10: 1, 30: -1, 80: -1, 101: 1} 5.0


## CPU QAOA criteria/readout

In [3]:
config=json.loads((ROOT/'configs/solvers/qaoa_cpu.json').read_text())
config['parameters']['layers']=2; config['optimizer']['parameters']['max_steps']=3
config['initialization']={'type':'fixed_angles','parameters':{'values':[0.3,0.2,0.1,0.4]}}
config['estimation']={'mode':'exact'}; config['readout']={'mode':'most_probable'}
config['limits']={'shots_total':0}; config['output']={'report':0}
q=QAOACPU(config); s=q.solve(graph); print(s,q.cut_value,q.resources)

{10: -1, 30: 1, 80: 1, 101: 1} 5.0 {'shots_used': 0, 'known_spent': 0, 'uncertain_upper_charge': 0, 'total_charged': 0, 'training_shots': 0, 'logical_training_shots': 0, 'readout_shots': 0, 'readout_reserved': 0, 'shots_available': 0, 'objective_evaluations': 27, 'optimizer_steps': 3, 'num_stages': 4, 'num_treads': 1, 'checkpoint_writes': 0}


## Recursive RQAOA (small residual)

In [4]:
rconfig=json.loads((ROOT/'configs/solvers/rqaoa_cpu.json').read_text())
rconfig['parameters']={'finish_at':1,'max_reductions':1}; rconfig['limits']={'shots_total':64}
rconfig['output']={'report':0}; rconfig['correlation_source']['parameters']['max_vertices']=4
rconfig['correlation_source']['estimation']={'mode':'exact'}; rconfig['correlation_source']['correlations']={'mode':'exact'}
r=RQAOACPU(rconfig); rs=r.solve(graph); print(rs,r.cut_value,r.reductions_completed)

{10: 1, 30: 1, 101: 1, 80: -1} 2.0 1


## Graph recipe and typed plan

In [5]:
g=GraphFactory.from_recipe(json.loads((ROOT/'configs/graphs/connected_path_fill_6.json').read_text()))
p=build_connected_bfs_partition(g,max_size=3,seed=2,merge_singletons=True)
a=analyze_plan(g,p); print(len(g),len(g.edges),p.plan_kind,a['vertex_coverage'],a['edge_coverage'])

6 9 vertex_partition 1.0 0.4444444444444444


## Edge-subset plan has an explicit edge domain

In [6]:
e=build_overlap_edge_partition(g,max_size=3,seed=0)
ea=analyze_plan(g,e); print(e.completion_status,ea['edge_coverage'],ea['edge_exclusive'])
print([(part.nodes,part.edge_domain,len(part.edges)) for part in e.parts[:2]])

complete 1.0 True
[((-7, 80, 101), 'explicit', 2), ((-7, 10, 205), 'explicit', 3)]


## Checkpoint/resume QAOA

In [7]:
checkpoint_config=dict(config); checkpoint_config['checkpoint']={'enabled':True,'path':tempfile.mkdtemp(prefix='qa_ckpt_'),'every':{'optimizer_iterations':1},'durable_resource_journal':True}
ck=QAOACPU(checkpoint_config); ck.solve(graph,control=RunControl(pause_after={'optimizer_iterations':1}))
print('paused',ck.status,ck.phase if hasattr(ck,'phase') else '',ck.last_checkpoint is not None)
resumed=restore(ck.last_checkpoint.path); resumed.resume(); print('resumed',resumed.status,resumed.reductions_completed if hasattr(resumed,'reductions_completed') else '')

paused paused  True
resumed completed 


## Release blockers

Batch/backtracking/hierarchical controllers and full 89-source closure are not claimed here; see `RELEASE_READINESS.md`.